# 📘 Notebook 06: Part-of-Speech Tagging and Named Entities

### Course: *From Words to Meaning: Natural Language Processing in Python*
**Instructor:** Ioannis Tsioulis

*Module B: The Structure of Language · Notebook 3 of 3 in this module · (6 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain what **parts of speech** are and why assigning them is harder than using a dictionary
- Build a **baseline tagger** from counts and evaluate it properly on unseen sentences
- Describe a **hidden Markov model** and estimate its transition and emission probabilities from data
- Implement the **Viterbi algorithm** by hand and explain why it is efficient
- Use NLTK's trained tagger and compare it with your own
- Explain what **named entity recognition** is and how it reduces to a tagging problem

> **Prerequisites:** Notebooks 03 and 05. Dictionaries and `Counter`. The idea of training and test data, covered in *From Python to Deep Learning & Fine-Tuning*, Notebook 08.
>
> 🔭 **Why this notebook matters:** this is the first system in the course that **learns from data**. We give it sentences that people have labelled and it works out how to label new ones. The task, giving each word a grammatical label, is modest. The method is not: counting examples, turning counts into probabilities, and searching for the most probable answer is the template for a great deal of what follows.

> ℹ️ **Setup note.** Run this cell once to download the data and models used below.

In [ ]:
import nltk

for package in ["treebank", "universal_tagset", "punkt", "punkt_tab",
                "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng",
                "maxent_ne_chunker", "maxent_ne_chunker_tab", "words"]:
    nltk.download(package, quiet=True)

import math
from collections import Counter

## 1. Parts of speech

### Intuition first
At school you learned to sort words into kinds: nouns name things, verbs describe actions, adjectives describe nouns. These kinds are called **parts of speech** (PoS). They matter because they determine how a word may be used. Knowing that a word is a noun tells you it can follow *the*, can be the subject of a verb, and can probably be made plural.

Notebook 03 already needed them: the lemmatiser could not tell whether *leaves* belonged to *leaf* or to *leave* without knowing the part of speech. And the grammars of Notebook 05 were written entirely in terms of categories like `N`, `V` and `Det`.

### Formal definition
**Part-of-speech tagging** is the task of assigning to each token in a sentence one label from a fixed set, the **tagset**. We will use the *universal* tagset, which has twelve tags.

| Tag | Meaning | Examples |
|---|---|---|
| NOUN | noun | *dog*, *London*, *idea* |
| VERB | verb | *run*, *is*, *could* |
| ADJ | adjective | *old*, *green* |
| ADV | adverb | *quickly*, *very*, *not* |
| PRON | pronoun | *she*, *it*, *who* |
| DET | determiner | *the*, *a*, *this* |
| ADP | preposition | *in*, *of*, *with* |
| NUM | number | *three*, *1987* |
| CONJ | conjunction | *and*, *but* |
| PRT | particle | *to* (before a verb), *up* in *give up*, *'s* |
| . | punctuation | *.* *,* *?* |
| X | other | foreign words, symbols |

We need sentences in which every word has already been tagged by a person. The Penn Treebank sample from Notebook 05 provides them.

In [ ]:
from nltk.corpus import treebank
from nltk.tag import map_tag

tagged = []
for sentence in treebank.tagged_sents():
    clean = []
    for word, tag in sentence:
        if tag != "-NONE-":                # skip the invisible markers used by the annotators
            clean.append((word, map_tag("en-ptb", "universal", tag)))
    tagged.append(clean)

print(len(tagged), "tagged sentences\n")
for word, tag in tagged[1]:
    print(f"{word:<12} {tag}")

### Why not just use a dictionary?
Because a great many words belong to more than one category. Consider *book*: *"I read a **book**"* (noun) and *"please **book** a table"* (verb). Let us measure how common this is in our data.

In [ ]:
tags_of_word = {}
total_tokens = 0
for sentence in tagged:
    for word, tag in sentence:
        word = word.lower()
        if word not in tags_of_word:
            tags_of_word[word] = Counter()
        tags_of_word[word][tag] += 1
        total_tokens += 1

ambiguous_types = 0
ambiguous_tokens = 0
for word, counts in tags_of_word.items():
    if len(counts) > 1:
        ambiguous_types += 1
        ambiguous_tokens += sum(counts.values())

print(f"Word types with more than one tag: {ambiguous_types / len(tags_of_word):.0%}")
print(f"Tokens that are such words:        {ambiguous_tokens / total_tokens:.0%}")
print()
for word in ["that", "back", "more", "close", "plans"]:
    print(f"{word:<7} {dict(tags_of_word[word])}")

Only a small share of the **vocabulary** is ambiguous, but those words are used so often that they make up a large share of the running **text**. Zipf's law again: the frequent words are the troublesome ones.

## 2. A baseline: the most frequent tag

### Intuition first
The numbers above also contain good news. *that* is ambiguous, but it is not evenly ambiguous: one of its tags is much more common than the others. So here is the simplest tagger imaginable. For each word, always answer with the tag it had most often in the training data.

A simple method like this is called a **baseline**. Its purpose is to tell us how hard the problem really is. Any cleverer method has to justify itself by beating it.

### Training and testing
We must not test the tagger on the sentences it learned from. That would be like setting an exam made of the homework questions. We keep the first 3,000 sentences for **training** and hold out the rest for **testing**.

In [ ]:
train = tagged[:3000]
test = tagged[3000:]

word_tag_counts = {}
for sentence in train:
    for word, tag in sentence:
        word = word.lower()
        if word not in word_tag_counts:
            word_tag_counts[word] = Counter()
        word_tag_counts[word][tag] += 1

most_frequent_tag = {}
for word, counts in word_tag_counts.items():
    most_frequent_tag[word] = counts.most_common(1)[0][0]

def baseline_tagger(words):
    tags = []
    for word in words:
        tags.append(most_frequent_tag.get(word.lower(), "NOUN"))    # unknown word: guess NOUN
    return tags

print(baseline_tagger("The old man saw the boats".split()))

A word the tagger has never seen is tagged `NOUN`, since new words are most often nouns. Now the measurement: the share of test tokens that receive the correct tag.

In [ ]:
def accuracy(tagger, data):
    correct = 0
    total = 0
    for sentence in data:
        words = [word for word, tag in sentence]
        gold = [tag for word, tag in sentence]
        predicted = tagger(words)
        for answer, truth in zip(predicted, gold):
            if answer == truth:
                correct += 1
            total += 1
    return correct / total

baseline_accuracy = accuracy(baseline_tagger, test)
print(f"Baseline accuracy on unseen sentences: {baseline_accuracy:.1%}")

A method with no understanding of grammar at all gets more than nine words in ten right. This is typical of language tasks, and it is a warning: a headline accuracy means little until you know the baseline.

Where do the mistakes come from? Let us separate the words the tagger saw in training from those it did not.

In [ ]:
known_correct = known_total = unknown_correct = unknown_total = 0

for sentence in test:
    words = [word for word, tag in sentence]
    predicted = baseline_tagger(words)
    for (word, truth), answer in zip(sentence, predicted):
        if word.lower() in most_frequent_tag:
            known_total += 1
            known_correct += (answer == truth)
        else:
            unknown_total += 1
            unknown_correct += (answer == truth)

print(f"Known words:   {known_total:>6,} tokens, accuracy {known_correct / known_total:.1%}")
print(f"Unknown words: {unknown_total:>6,} tokens, accuracy {unknown_correct / unknown_total:.1%}")

Two weaknesses are now visible. Unknown words are handled badly. And for known words the tagger ignores the **sentence**: it gives *book* the same tag after *a* as after *please*. The next model addresses the second weakness directly.

## 3. Using context: the hidden Markov model

### Intuition first
Tags follow each other in regular ways. After a determiner (*the*) comes a noun or an adjective, almost never a verb. After a pronoun (*she*) a verb is very likely. A tagger that knows this can reason: *"this word could be a noun or a verb, but the previous word was 'the', so it is surely a noun."*

A **hidden Markov model (HMM)** combines exactly two kinds of evidence:

- **How likely is this tag after the previous tag?** (the grammar of tag sequences)
- **How likely is this word, given this tag?** (the dictionary)

The picture behind the name: imagine the sentence was produced by a machine that hops from tag to tag, and at each stop emits a word. We see only the words. The tags are **hidden**, and our job is to recover them.

### Formal definition
An HMM tagger has two tables of probabilities, both estimated by counting in the training data:

- **Transition probabilities** $P(t_i \mid t_{i-1})$: of all the times tag $t_{i-1}$ occurred, how often was it followed by $t_i$?
- **Emission probabilities** $P(w_i \mid t_i)$: of all the times tag $t_i$ occurred, how often was the word $w_i$?

The probability of a sentence together with a tag sequence is the product of all of these:

$$P(w_1 \dots w_n, t_1 \dots t_n) = \prod_{i=1}^{n} P(t_i \mid t_{i-1}) \; P(w_i \mid t_i)$$

The model assumes each tag depends only on the one before it. That is the **Markov assumption**, a deliberate simplification that makes the counting and the search feasible.

Let us count. We add an artificial tag `<s>` before each sentence, so that we also learn which tags tend to start a sentence.

In [ ]:
transition_counts = {}        # transition_counts[previous_tag][tag]
emission_counts = {}          # emission_counts[tag][word]

for sentence in train:
    previous = "<s>"
    for word, tag in sentence:
        word = word.lower()
        if previous not in transition_counts:
            transition_counts[previous] = Counter()
        transition_counts[previous][tag] += 1
        if tag not in emission_counts:
            emission_counts[tag] = Counter()
        emission_counts[tag][word] += 1
        previous = tag

tags = sorted(emission_counts)
print(tags)

In [ ]:
print("After a determiner, the next tag is:")
total = sum(transition_counts["DET"].values())
for tag, count in transition_counts["DET"].most_common(4):
    print(f"  {tag:<5} {count / total:.2f}")

print("\nAfter a pronoun, the next tag is:")
total = sum(transition_counts["PRON"].values())
for tag, count in transition_counts["PRON"].most_common(4):
    print(f"  {tag:<5} {count / total:.2f}")

The counts confirm the intuition: a determiner is followed by a noun or an adjective in the great majority of cases, and the most likely tag after a pronoun is a verb. Nobody wrote these rules. They were read off the data.

### Two practical details
**Unseen events.** If a tag pair never occurred in training, its counted probability is zero, and a single zero wipes out the whole product. We add a small amount to every count so that nothing is ever exactly impossible. This is called **smoothing**, and Notebook 07 studies it properly.

**Unknown words.** For a word never seen in training we have no emission count at all. A neat trick is to assume that unknown words behave like **rare** words, and that the **spelling** of a word is a clue to its tag. So we take the words that occurred exactly once in training, sort them by a rough description of their shape, and see which tags each shape had.

In [ ]:
def shape(word):
    if any(character.isdigit() for character in word):
        return "<number>"
    if word.endswith("ly"):
        return "<-ly>"
    if word.endswith("ing"):
        return "<-ing>"
    if word.endswith("ed"):
        return "<-ed>"
    if word.endswith("s"):
        return "<-s>"
    if word[0].isupper():
        return "<Capital>"
    return "<other>"

rare_shape_tags = {}                  # rare_shape_tags[shape][tag]
for sentence in train:
    for word, tag in sentence:
        if sum(word_tag_counts[word.lower()].values()) == 1:
            if shape(word) not in rare_shape_tags:
                rare_shape_tags[shape(word)] = Counter()
            rare_shape_tags[shape(word)][tag] += 1

for word_shape, counts in rare_shape_tags.items():
    total = sum(counts.values())
    summary = ", ".join(f"{tag} {count / total:.2f}" for tag, count in counts.most_common(3))
    print(f"{word_shape:<10} {total:>5} rare words   {summary}")

The shapes are informative. A rare word containing a digit is almost always a number, one ending in *-ly* is usually an adverb, and one ending in *-ing* or *-ed* is most often a verb. Rare words are practically never determiners or pronouns, because those are small closed sets of very frequent words. When the tagger meets an unknown word, it will treat it as "a rare word of this shape".

Now we turn the counts into **log**-probabilities. A product of many small numbers quickly becomes too small for the computer to represent, so we add logarithms and never multiply the probabilities themselves.

In [ ]:
tag_totals = {tag: sum(emission_counts[tag].values()) for tag in tags}

log_transition = {}
for previous in ["<s>"] + tags:
    counts = transition_counts.get(previous, Counter())
    total = sum(counts.values())
    log_transition[previous] = {}
    for tag in tags:
        log_transition[previous][tag] = math.log((counts[tag] + 1) / (total + len(tags)))

def log_emission(tag, word):
    if word.lower() not in most_frequent_tag:         # a word never seen in training
        count = rare_shape_tags[shape(word)][tag]
        return math.log((count + 0.1) / (tag_totals[tag] + len(tags)))
    count = emission_counts[tag][word.lower()]
    if count == 0:
        return -math.inf                              # a known word that never had this tag
    return math.log(count / tag_totals[tag])

## 4. The Viterbi algorithm

### Intuition first
We have a way to **score** any tag sequence for a sentence. We want the sequence with the highest score. Trying them all is hopeless: with 12 tags, a sentence of 20 words has $12^{20}$ possible sequences, a number with 22 digits.

The key observation is that we do not need to. Suppose we already know, for the word at position $i-1$, the best possible score of any sequence ending in each of the 12 tags. Then for position $i$, the best sequence ending in a given tag must extend **one of those twelve**. Everything before that is already decided. So we move through the sentence one word at a time, keeping only 12 numbers per position, and at the end we trace back the choices that led to the best final score.

This is the same idea as the CKY table in Notebook 05: solve small problems once and reuse them. It is named after Andrew Viterbi.

### Formal definition
Let $v_i(t)$ be the score of the best tag sequence for the first $i$ words that ends in tag $t$. Then

$$v_i(t) = \max_{t'} \big[\, v_{i-1}(t') + \log P(t \mid t') \,\big] + \log P(w_i \mid t)$$

and we remember, for each $t$, which previous tag $t'$ achieved the maximum.

In [ ]:
def viterbi_tagger(words):
    best = [{}]             # best[i][tag]: score of the best path ending in tag at position i
    came_from = [{}]         # came_from[i][tag]: the previous tag on that path

    for tag in tags:
        best[0][tag] = log_transition["<s>"][tag] + log_emission(tag, words[0])

    for i in range(1, len(words)):
        best.append({})
        came_from.append({})
        for tag in tags:
            best_score = -math.inf
            best_previous = tags[0]
            for previous in tags:
                score = best[i - 1][previous] + log_transition[previous][tag]
                if score > best_score:
                    best_score = score
                    best_previous = previous
            best[i][tag] = best_score + log_emission(tag, words[i])
            came_from[i][tag] = best_previous

    # find the best final tag, then walk backwards
    last = max(tags, key=lambda tag: best[-1][tag])
    path = [last]
    for i in range(len(words) - 1, 0, -1):
        last = came_from[i][last]
        path.append(last)
    path.reverse()
    return path

sentence = "Please book the old book".split()

print(f"{'word':<8} {'baseline':<9} {'HMM'}")
for word, base, hmm in zip(sentence, baseline_tagger(sentence), viterbi_tagger(sentence)):
    print(f"{word:<8} {base:<9} {hmm}")

The baseline gives both occurrences of *book* the same tag, as it must. The HMM tags the first as a verb and the second as a noun, because it weighs each candidate tag against the tag before it.

For a sentence of $n$ words and $T$ tags the algorithm does about $n \times T^2$ steps. For our 20-word sentence that is under 3,000 steps in place of $12^{20}$.

Now the test that matters: is the HMM better than the baseline on sentences it has never seen?

In [ ]:
hmm_accuracy = accuracy(viterbi_tagger, test)

print(f"Baseline: {baseline_accuracy:.1%}")
print(f"HMM:      {hmm_accuracy:.1%}")
print(f"Errors removed: {1 - (1 - hmm_accuracy) / (1 - baseline_accuracy):.0%}")

The gain looks small in percentage points and is large where it counts: about a third of the baseline's mistakes have disappeared. When accuracy is already above 90%, the honest way to compare two systems is by how many of the **remaining errors** each one removes.

> ⚠️ **Common pitfalls**
>
> - Multiplying probabilities directly. Long products underflow to zero. Work with **sums of logarithms**.
> - Evaluating on the training data. Our tagger has memorised those sentences, and its accuracy there says nothing about new text.
> - Forgetting that the test data resembles the training data. All our sentences are from one newspaper. On tweets, medical notes or poetry, accuracy would fall noticeably. A model is only as general as the data it learned from.

## 5. A professional tagger

NLTK ships a tagger trained on much more data, using a richer model that looks at neighbouring words, prefixes, suffixes and capital letters. Its interface is one function.

In [ ]:
sentence = "They refuse to permit us to obtain the refuse permit".split()

print(f"{'word':<8} {'baseline':<9} {'our HMM':<8} {'NLTK'}")
rows = zip(sentence, baseline_tagger(sentence), viterbi_tagger(sentence), nltk.pos_tag(sentence, tagset="universal"))
for word, base, hmm, (same_word, professional) in rows:
    print(f"{word:<8} {base:<9} {hmm:<8} {professional}")

The sentence uses *refuse* and *permit* twice each, first as verbs and then as nouns. NLTK's tagger gets all four right. Our HMM does not, and the reason is instructive. In our 3,000 training sentences *refuse* and *permit* happened to occur only as verbs, so the model's emission probability for them as nouns is exactly zero, and no amount of context can outweigh a zero. A model cannot predict what its training data gave it no reason to consider. More data is the cure, and NLTK's tagger had far more.

In [ ]:
def nltk_tagger(words):
    return [tag for word, tag in nltk.pos_tag(words, tagset="universal")]

print(f"NLTK tagger on our test sentences: {accuracy(nltk_tagger, test[:300]):.1%}")

> 🧠 Treat that last number with suspicion. NLTK's tagger was trained on Wall Street Journal text, quite possibly including these very sentences, so this is not a fair test of it. The broader point stands, though: taggers for English newspaper text reach about 97% accuracy, which is close to the rate at which human annotators agree with one another. The remaining errors are concentrated in exactly the hard, ambiguous cases.

## 6. Named entity recognition

### Intuition first
Some of the most useful information in a text is in its names: who, where, which company, when. Finding them is the first step of almost every application that extracts facts from text, from search engines to systems that read contracts or medical records.

### Formal definition
**Named entity recognition (NER)** is the task of finding the spans of text that refer to real-world entities and classifying each one, typically as a PERSON, an ORGANIZATION, a location (in NLTK: GPE, a *geo-political entity*), a date, and so on.

A first attempt by hand: names are capitalised.

In [ ]:
import re

text = "Yesterday Maria Papadaki flew from Athens to London to meet the directors of Rolls Royce. The meeting was organised by the European Commission."

print(re.findall(r"[A-Z][a-z]+(?: [A-Z][a-z]+)*", text))

The pattern finds the names, and also `Yesterday` and `The`, which are capitalised only because they begin a sentence. It tells us nothing about the **type** of each entity. And it would fail completely on text written in lower case, or in a language such as German where every noun is capitalised.

NLTK provides a trained recogniser. It works on top of the PoS tags, which is why the two tasks appear in the same notebook.

In [ ]:
tokens = nltk.word_tokenize(text)
tree = nltk.ne_chunk(nltk.pos_tag(tokens))

for subtree in tree:
    if hasattr(subtree, "label"):
        name = " ".join(word for word, tag in subtree.leaves())
        print(f"{subtree.label():<13} {name}")

The recogniser returns a shallow tree in which each entity is a small labelled group of tokens. Study the result with a critical eye. It no longer mistakes `Yesterday` for a name, and it types the people and the cities correctly. When this notebook was written it also labelled the company *Rolls Royce* as a person. Recognisers of this generation do well on common names of people and places, and are noticeably less reliable on organisations and on names they have not seen.

### NER is a tagging problem
How would you train such a system? By turning it into the problem we just solved. Each token gets a tag saying whether it is at the **B**eginning of an entity, **I**nside one, or **O**utside any entity. This is the **IOB** scheme.

In [ ]:
from nltk.chunk import tree2conlltags

for word, pos, entity_tag in tree2conlltags(tree)[:14]:
    print(f"{word:<10} {pos:<5} {entity_tag}")

`B-PERSON` followed by `I-PERSON` marks a two-word name of a person. With this encoding, recognising entities becomes: *give every token one tag from a fixed set, using the context*. That is exactly what our HMM does. The same recipe (annotated examples, a sequence model, a search for the best tag sequence) covers PoS tagging, NER, and a long list of other tasks. This is why sequence labelling is one of the central ideas of NLP.

---
## ✏️ Exercises

### Exercise 1 (The most ambiguous words)
Using `tags_of_word` from Section 1, find all the words that occur with **four or more** different tags. Print each with its tag counts. What kind of words are they?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for word, counts in tags_of_word.items():
    if len(counts) >= 4:
        print(f"{word:<8} {dict(counts)}")
```

*They are short, very frequent words such as "in", "on", "down", "off" and "back", which serve as prepositions, particles and adverbs. The words used most often are the ones with the most uses. Look also at "the": a handful of its several thousand occurrences carry odd tags. Those are slips by the human annotators. Real training data always contains some noise.*
</details>

### Exercise 2 (Starting a sentence)
Using `transition_counts`, print the probability of each tag at the **start** of a sentence, most likely first.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
total = sum(transition_counts["<s>"].values())
for tag, count in transition_counts["<s>"].most_common():
    print(f"{tag:<5} {count / total:.3f}")
```

*Sentences in this corpus most often begin with a noun (often a name), a determiner or a preposition, and hardly ever with a particle. The artificial `<s>` tag lets the model learn this with no extra machinery.*
</details>

### Exercise 3 (Smarter guesses for unknown words)
Our HMM improved on the baseline in two ways at once: it uses context, and it guesses unknown words from their shape. How much of the gain is due to each? Give the **baseline** the same advantage. Write `guess_tag(word)` for unknown words: a digit suggests `NUM`, the ending `-ly` suggests `ADV`, the endings `-ing` and `-ed` suggest `VERB`, and anything else is a `NOUN`. Build a new tagger that uses it, and compare all three accuracies.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def guess_tag(word):
    if any(character.isdigit() for character in word):
        return "NUM"
    if word.endswith("ly"):
        return "ADV"
    if word.endswith("ing") or word.endswith("ed"):
        return "VERB"
    return "NOUN"

def better_baseline(words):
    result = []
    for word in words:
        if word.lower() in most_frequent_tag:
            result.append(most_frequent_tag[word.lower()])
        else:
            result.append(guess_tag(word))
    return result

print(f"Original baseline: {baseline_accuracy:.1%}")
print(f"With guessing:     {accuracy(better_baseline, test):.1%}")
print(f"HMM:               {hmm_accuracy:.1%}")
```

*Most of the improvement came from handling unknown words sensibly, and a smaller part from context. This kind of experiment, removing one ingredient at a time to see what each contributes, is called an ablation. Without it we would have given the credit to the wrong idea.*
</details>

### Exercise 4 (Where does the HMM go wrong?)
Run `viterbi_tagger` over the test set and count each kind of mistake as a pair *(correct tag, predicted tag)*. Print the eight most common confusions.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
confusions = Counter()
for sentence in test:
    words = [word for word, tag in sentence]
    predicted = viterbi_tagger(words)
    for (word, truth), answer in zip(sentence, predicted):
        if answer != truth:
            confusions[(truth, answer)] += 1

for (truth, answer), count in confusions.most_common(8):
    print(f"correct {truth:<5} predicted {answer:<5} {count}")
```

*The errors are not random. They cluster around a few pairs that are hard for people too, such as nouns against adjectives ("a gold watch") and verbs against nouns. Looking at the errors, and not only at the accuracy, is how you find out what to improve.*
</details>

### Exercise 5 (How much data is enough?)
Train the **baseline** on the first 100, 300, 1,000 and 3,000 training sentences, and print its accuracy on the test set each time. (You will need to rebuild the dictionary of most frequent tags inside a function.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def train_baseline(sentences):
    counts = {}
    for sentence in sentences:
        for word, tag in sentence:
            word = word.lower()
            if word not in counts:
                counts[word] = Counter()
            counts[word][tag] += 1
    lookup = {word: tag_counts.most_common(1)[0][0] for word, tag_counts in counts.items()}

    def tagger(words):
        return [lookup.get(word.lower(), "NOUN") for word in words]
    return tagger

for size in [100, 300, 1000, 3000]:
    tagger = train_baseline(train[:size])
    print(f"{size:>5} sentences: {accuracy(tagger, test):.1%}")
```

*Accuracy climbs quickly at first and then more and more slowly: each tenfold increase in data buys a smaller gain. This curve of diminishing returns appears in almost every learning system, and it follows from Zipf's law: after the frequent words have been seen, new data mostly brings rare ones.*
</details>

### Exercise 6 (Entities in a news story, a little harder)
Write a function `entities(text)` that returns a list of `(label, name)` pairs for all named entities in a text of several sentences. Use `nltk.sent_tokenize` to process one sentence at a time. Try it on the paragraph below and note one correct result and one mistake.

In [ ]:
story = """Angela Merkel met Emmanuel Macron in Paris on Tuesday. The two leaders discussed
the future of the European Union with officials from Microsoft and Siemens. Later Macron
travelled to Berlin, where he gave a speech at Humboldt University."""
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
story = """Angela Merkel met Emmanuel Macron in Paris on Tuesday. The two leaders discussed
the future of the European Union with officials from Microsoft and Siemens. Later Macron
travelled to Berlin, where he gave a speech at Humboldt University."""

def entities(text):
    found = []
    for sentence in nltk.sent_tokenize(text):
        tree = nltk.ne_chunk(nltk.pos_tag(nltk.word_tokenize(sentence)))
        for subtree in tree:
            if hasattr(subtree, "label"):
                name = " ".join(word for word, tag in subtree.leaves())
                found.append((subtree.label(), name))
    return found

for label, name in entities(story):
    print(f"{label:<13} {name}")
```

*Check each line against what you know. The people and cities are usually found. Company names and the first word of a sentence are where this recogniser tends to slip, labelling an organisation as a person or a place, or the reverse. Modern recognisers built on the neural models of Module E make far fewer such errors, but the task, and the IOB encoding, are the same.*
</details>

## 🔑 Key takeaways

- **PoS tagging** assigns each token a grammatical category. It is hard because frequent words are **ambiguous**.
- Always build a **baseline** first. "Most frequent tag" is already above 90%, so accuracy figures mean little without it.
- Evaluate on **held-out** data, and look separately at **unknown words** and at the kinds of error.
- An **HMM** combines **transition** probabilities (tag after tag) and **emission** probabilities (word given tag), both obtained by counting.
- The **Viterbi algorithm** finds the best tag sequence in about $n \times T^2$ steps by reusing partial results.
- **Smoothing** and a sensible treatment of **unknown words** are necessary in any model built from counts.
- **Named entity recognition** finds and classifies names. With the **IOB** encoding it becomes a tagging problem like any other.

### 🏁 You have completed Module B: The Structure of Language!
You have seen what machines are needed for the structure of language, written grammars and parsed with them, discovered how pervasive ambiguity is, and built a tagger that learns from annotated examples. The lesson of the module is the one stated at the end of Notebook 05: rules written by hand run out, and counting takes over.

**Next:** *Module C: Counting Words*, starting with *Notebook 07: N-gram Language Models*, where counting is used to predict the next word.

---
*End of Module B.*

---
*© Ioannis Tsioulis. *From Words to Meaning: Natural Language Processing in Python*. Prepared for educational use.*